In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler , LabelEncoder , OneHotEncoder
import pickle

In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## Preprocessin Data

data = data.drop(['RowNumber' , 'CustomerId' , 'Surname'] , axis = 1)

In [4]:
## Encode categorical varibales

label_encode_gen = LabelEncoder()
data['Gender'] = label_encode_gen.fit_transform(data['Gender'])

In [5]:
## One-hot ENcode for geography

onehot_encode_geo = OneHotEncoder(handle_unknown='ignore')

geo_encode = onehot_encode_geo.fit_transform(data[['Geography']]).toarray()
geo_encode_df = pd.DataFrame(geo_encode , columns = onehot_encode_geo.get_feature_names_out(['Geography']))
geo_encode_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [6]:
## Combining our onehot_encoded df with data

data = pd.concat([data.drop('Geography' , axis = 1) , geo_encode_df] , axis  = 1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [7]:
### Independent and dependent features

X = data.drop('EstimatedSalary' , axis = 1)
y = data['EstimatedSalary']

In [10]:
## Splitting our taining and testing data

X_train , X_test , y_train , y_test = train_test_split(X , y , test_size = 0.2 , random_state = 42)


In [11]:
## Scaling our training features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
### Saving our encoders and scalers for later loading

with open('label_encode_gen.pkl' , 'wb') as file:
    pickle.dump(label_encode_gen , file)

with open('onehot_encode_geo.pkl' , 'wb') as file:
    pickle.dump(onehot_encode_geo , file)

with open('scaler.pkl' , 'wb') as file:
    pickle.dump(scaler , file)

## Training our ANN with Regression 

In [13]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense


In [14]:
## Building model
model = Sequential([
    Dense(64 , activation='relu' , input_dim = (X_train.shape[1])),
    Dense(32 , activation='relu'),
    Dense(1 , activation='linear')    # output layer for regression
])

d:\Udemy GenAI Course\ANN classification\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [16]:
model.compile(optimizer = 'adam' , loss = 'mean_absolute_error' , metrics = ['mae'])

In [18]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [29]:
## training model

from tensorflow.keras.callbacks import EarlyStopping , TensorBoard
import datetime

# Set up TensorBoard
log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir = log_dir , histogram_freq = 1)

In [30]:
## setting up earlystopping
early = EarlyStopping(monitor = 'val_loss' , patience = 10 , restore_best_weights = True)

In [31]:
history = model.fit(X_train , y_train , validation_data = [X_test , y_test] , epochs = 100 , callbacks=[early , tensorboard_callback])

Epoch 1/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49486.6172 - mae: 49486.6172 - val_loss: 50266.8984 - val_mae: 50266.8984
Epoch 2/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49478.1914 - mae: 49478.1914 - val_loss: 50254.5039 - val_mae: 50254.5039
Epoch 3/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 49474.6602 - mae: 49474.6602 - val_loss: 50253.1562 - val_mae: 50253.1562
Epoch 4/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49464.3672 - mae: 49464.3672 - val_loss: 50266.1875 - val_mae: 50266.1875
Epoch 5/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49463.0352 - mae: 49463.0352 - val_loss: 50272.5117 - val_mae: 50272.5117
Epoch 6/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49451.5898 - mae: 49451.5898 - val_loss: 50255.8906 - val_mae: 50255.8906
Epoch 7/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 49446.8711 - mae: 49446.8711 - val_loss: 50269.8398 - val_mae: 50269.8398
Epoch 8/100
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - 

In [32]:
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [34]:
%tensorboard --logdir regressionlogs/fit

In [39]:
## Model Evaluation
model.save('reg-model.h5')

In [38]:
## Evaluation for test data

test_loss = test_mae = model.evaluate(X_test , y_test)
print(f"Test MAE : {test_mae}")

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 50253.1562 - mae: 50253.1562
Test MAE : [50253.15625, 50253.15625]
